# 05 · 训练与评测基线：从 loss 到可解释指标

自动驾驶岗位不只要求“会训练一个模型”，还要求你能说明数据划分、阈值、类别不平衡、校准和失败案例。本 notebook 用纯 NumPy 实现一个小型二分类 MLP，练习一套可迁移到感知模型的实验闭环。

学习目标：

- 只用训练集统计量做归一化，避免验证集信息泄漏。
- 从 forward、backward、loss 曲线走到 accuracy、precision、recall、F1 和 ECE。
- 通过阈值滑块观察安全任务中 precision–recall 的取舍。
- 记录随机种子、训练时间和失败样本，而不是只保留一个最终分数。

这个 toy data 不是自动驾驶 benchmark；它是训练/评测接口的最小实验。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter
from ipywidgets import interact, FloatSlider

rng = np.random.default_rng(7)
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True

n = 1800
angles = rng.uniform(0, 2 * np.pi, n)
radii = np.where(np.arange(n) % 2 == 0, rng.normal(0.75, 0.12, n), rng.normal(1.55, 0.14, n))
X = np.c_[radii * np.cos(angles), radii * np.sin(angles)]
y = (radii > 1.05).astype(float)

perm = rng.permutation(n)
train_end, valid_end = int(0.65 * n), int(0.82 * n)
train_idx, valid_idx, test_idx = perm[:train_end], perm[train_end:valid_end], perm[valid_end:]
mean = X[train_idx].mean(axis=0)
std = X[train_idx].std(axis=0) + 1e-6
Xn = (X - mean) / std
X_train, y_train = Xn[train_idx], y[train_idx]
X_valid, y_valid = Xn[valid_idx], y[valid_idx]
X_test, y_test = Xn[test_idx], y[test_idx]
print(f'train/valid/test = {len(train_idx)}/{len(valid_idx)}/{len(test_idx)}')
print(f'train-only mean={mean.round(3)}, std={std.round(3)}')


## Part A — 一个可复现的 MLP 训练循环

模型只有一个隐藏层，但训练接口已经包含真实项目中最重要的几件事：初始化、forward、反向传播、验证集监控和 best checkpoint 选择。

练习时可以改变 hidden_dim、learning_rate 和 epochs，观察欠拟合、过拟合与训练不稳定。


In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def train_mlp(hidden_dim=24, learning_rate=0.08, epochs=350, seed=11):
    local_rng = np.random.default_rng(seed)
    W1 = local_rng.normal(0, 0.35, size=(2, hidden_dim))
    b1 = np.zeros(hidden_dim)
    W2 = local_rng.normal(0, 0.35, size=(hidden_dim, 1))
    b2 = np.zeros(1)
    history = []
    best = None
    best_valid = np.inf

    for epoch in range(epochs):
        z1 = X_train @ W1 + b1
        h = np.maximum(z1, 0)
        logits = h @ W2 + b2
        p = sigmoid(logits[:, 0])
        eps = 1e-7
        loss = -np.mean(y_train * np.log(p + eps) + (1 - y_train) * np.log(1 - p + eps))

        dlogits = (p - y_train)[:, None] / len(X_train)
        dW2 = h.T @ dlogits
        db2 = dlogits.sum(axis=0)
        dh = dlogits @ W2.T
        dz1 = dh * (z1 > 0)
        dW1 = X_train.T @ dz1
        db1 = dz1.sum(axis=0)
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        valid_p = sigmoid(np.maximum(X_valid @ W1 + b1, 0) @ W2 + b2)[:, 0]
        valid_loss = -np.mean(y_valid * np.log(valid_p + eps) + (1 - y_valid) * np.log(1 - valid_p + eps))
        history.append((loss, valid_loss))
        if valid_loss < best_valid:
            best_valid = valid_loss
            best = (W1.copy(), b1.copy(), W2.copy(), b2.copy())

    def predict_prob(features):
        w1, bb1, w2, bb2 = best
        return sigmoid(np.maximum(features @ w1 + bb1, 0) @ w2 + bb2)[:, 0]

    return predict_prob, np.asarray(history)

predict_prob, history = train_mlp()
print(f'best validation loss = {history[:, 1].min():.4f}')


In [ ]:
def classification_metrics(prob, labels, threshold=0.5):
    pred = prob >= threshold
    tp = np.sum(pred & (labels == 1))
    tn = np.sum(~pred & (labels == 0))
    fp = np.sum(pred & (labels == 0))
    fn = np.sum(~pred & (labels == 1))
    accuracy = (tp + tn) / max(len(labels), 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)}

def expected_calibration_error(prob, labels, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        mask = (prob >= low) & (prob < high if high < 1 else prob <= high)
        if mask.any():
            confidence = prob[mask].mean()
            accuracy = labels[mask].mean()
            ece += mask.mean() * abs(confidence - accuracy)
    return ece

test_prob = predict_prob(X_test)
print(classification_metrics(test_prob, y_test))
print(f'ECE = {expected_calibration_error(test_prob, y_test):.4f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history[:, 0], label='train loss')
ax[0].plot(history[:, 1], label='valid loss')
ax[0].set_title('training curve')
ax[0].set_xlabel('epoch')
ax[0].legend()
ax[1].scatter(X_test[:, 0], X_test[:, 1], c=test_prob, cmap='coolwarm', s=14)
ax[1].set_title('test probability')
ax[1].set_xlabel('normalized feature 0')
ax[1].set_ylabel('normalized feature 1')
plt.tight_layout()


### 交互练习：阈值是系统设计的一部分

在 corner-case 或行人检测中，漏检和误报的代价通常不对称。拖动阈值，观察 precision、recall 和 F1 如何变化。

请记录一个你愿意部署的 operating point，并说明为什么不能只报告 accuracy。


In [ ]:
def show_threshold(threshold=0.5):
    metrics = classification_metrics(test_prob, y_test, threshold)
    thresholds = np.linspace(0.05, 0.95, 91)
    precision, recall = [], []
    for value in thresholds:
        result = classification_metrics(test_prob, y_test, value)
        precision.append(result['precision'])
        recall.append(result['recall'])
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(thresholds, precision, label='precision')
    ax[0].plot(thresholds, recall, label='recall')
    ax[0].axvline(threshold, color='black', linestyle='--')
    ax[0].set_title(f'threshold={threshold:.2f}, F1={metrics["f1"]:.3f}')
    ax[0].set_xlabel('decision threshold')
    ax[0].legend()
    ax[1].bar(['TP', 'TN', 'FP', 'FN'], [metrics['tp'], metrics['tn'], metrics['fp'], metrics['fn']])
    ax[1].set_title(str(metrics))
    plt.tight_layout()
    plt.show()

interact(show_threshold, threshold=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5));


## 完成标准

- 对比至少两个 hidden_dim 或 learning_rate，保留训练曲线。
- 报告 test accuracy、precision、recall、F1、ECE 和混淆矩阵。
- 解释为什么归一化统计量必须来自 train split。
- 构造一个失败样本切片，并说明它更像 data problem、model problem 还是 threshold problem。
- 下一步把二分类接口替换成 3D detection 的 score、IoU 和 recall。
